# RSP Setup Check

Use this notebook first on Rubin Science Platform. It verifies the local checkout, persistent storage paths, package imports, and a tiny LSST-only ANTARES probe before any long backfill.

In [ ]:
from pathlib import Path
import os
import sys

# Make imports robust whether Jupyter starts in the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
candidate = Path.home() / "notebooks" / "ANTARES_Analysis"
if not (PROJECT_ROOT / "src").exists() and (candidate / "src").exists():
    PROJECT_ROOT = candidate
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, history, query, rsp_permissions

# Cooperative shared-workflow default: new files keep group write permission.
os.umask(0o002)

DATA_ROOT = config.DATA_ROOT
CACHE_ROOT = config.CACHE_ROOT
LSST_DATA_ROOT = config.LSST_ONLY_ROOT
NIGHTLY_ROOT = config.NIGHTLY_ROOT
CUMULATIVE_ROOT = config.CUMULATIVE_ROOT
ANALYSIS_ROOT = config.ANALYSIS_ROOT

print(f"Project root : {PROJECT_ROOT}")
print(f"HOME         : {os.getenv('HOME')}")
print(f"USER         : {os.getenv('USER') or os.getenv('JUPYTERHUB_USER')}")
print(f"SCRATCH_DIR  : {os.getenv('SCRATCH_DIR', 'not set')}")
print(f"Data root    : {DATA_ROOT}")
print(f"Cache root   : {CACHE_ROOT}")
print(f"Nightly root : {NIGHTLY_ROOT}")
print(f"Cumulative   : {CUMULATIVE_ROOT}")
print(f"Analysis root: {ANALYSIS_ROOT}")
print(f"Shared group : {config.EXPECTED_SHARED_GROUP}")


In [ ]:
import pandas as pd
import pyarrow
from antares_client.search import search as antares_search

print("Imports complete.")
print(f"pandas  : {pd.__version__}")
print(f"pyarrow : {pyarrow.__version__}")
config.print_config_summary()


In [ ]:
# Mandatory shared-root preflight before any ANTARES query or production write.
preflight_report = rsp_permissions.require_shared_data_root(
    DATA_ROOT,
    expected_group=config.EXPECTED_SHARED_GROUP,
)


In [ ]:
SCRATCH_ROOT = Path(os.getenv("SCRATCH_DIR", f"/scratch/{os.getenv('USER') or os.getenv('JUPYTERHUB_USER') or 'unknown_user'}")) / "ANTARES_Analysis"
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

for path in [
    CACHE_ROOT,
    NIGHTLY_ROOT,
    CUMULATIVE_ROOT,
    config.LSST_ONLY_ROOT / "analysis",
    ANALYSIS_ROOT,
    DATA_ROOT / "logs",
]:
    rsp_permissions.ensure_group_shared_path(path, expected_group=config.EXPECTED_SHARED_GROUP)
    print(f"OK: {path}")

print(f"Persistent LSST store: {LSST_DATA_ROOT}")
print(f"Shared cache root    : {CACHE_ROOT}")
print(f"Temporary scratch    : {SCRATCH_ROOT}")


In [ ]:
probe = query.query_range(
    label='RSP LSST-only probe',
    mjd_min=config.LSST_HISTORY_START_MJD,
    mjd_max=config.LSST_HISTORY_START_MJD + 1,
    n_samples=5,
    tag=config.QUERY_TAG,
    seed=None,
    verbose=True,
    lsst_only=True,
)

counts = query.lsst_identifier_counts(probe)
print(counts)
if not probe.empty:
    assert counts['lsst_identifier_count'] == len(probe), 'Probe returned non-LSST loci.'
    display_cols = [col for col in ['locus_id', 'ra', 'dec', 'newest_alert_observation_time', 'survey', 'ztf_object_id'] if col in probe.columns]
    display(probe[display_cols].head())
else:
    print('Probe returned 0 rows. Try a later MJD window before running backfill.')